# exp151_tvt_dense_addonly_confidence_features_on_exp092 train

Train-side add-only audit for target-free well-local tvt_dense confidence features on the exp092 U-projection LightGBM surface.

## Contents

1. Setup and configuration
2. Input and dense confidence feature contract
3. Train tvt_dense confidence variant
4. Metrics and generated artifacts

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from tvt_dense_addonly_confidence_features_on_exp092 import (
    FULL_REPLAY_TRAIN_FEATURES,
    OUTPUT_PREFIX,
    find_artifact,
    run_tvt_dense_addonly_confidence_features_on_exp092,
)

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Route:", cfg_get(config, "experiment.route"))
print("Mode:", cfg_get(config, "audit.mode"))
print("Parent:", cfg_get(config, "lineage.parent"))
print("Kernel sources:", cfg_get(config, "runtime.kaggle.kernel_sources"))
print("Active modes:", cfg_get(config, "model.training.active_modes"))
print("Active variants:", [v["name"] for v in cfg_get(config, "model.feature_ablation.active_variants", [])])


## 2. Input and dense confidence feature contract

In [ ]:
cache_path = find_artifact(
    FULL_REPLAY_TRAIN_FEATURES,
    cfg_get(config, "data.exp072_train_feature_cache_local"),
)
print("exp072 full replay train cache:", cache_path)

base_preview = pd.read_csv(cache_path, nrows=5, dtype={"id": str, "well": str})
preview_cols = [
    c
    for c in [
        "id",
        "well",
        "target",
        "last_known_tvt",
        "z",
        "md_since",
        "pf_ancc",
        "beam_mean_d",
        "likpf_mean_d",
        "tvt_dense_d",
        "tvt_densew_d",
        "tvt_dense50_d",
        "dense_std",
        "pf_vs_dense",
    ]
    if c in base_preview.columns
]
display(base_preview[preview_cols])
print("TVT dense confidence candidates:", list(cfg_get(config, "model.tvt_dense_confidence.dense_candidates", {}).keys()))


## 3. Train tvt_dense confidence variant

In [ ]:
summary = run_tvt_dense_addonly_confidence_features_on_exp092(
    output_dir=paths.artifacts_dir,
    train_dir=paths.train_data_dir,
    cache_path=cfg_get(config, "data.exp072_train_feature_cache_local"),
    projection_config=cfg_get(config, "model.u_projection", {}),
    dense_config=cfg_get(config, "model.tvt_dense_confidence", {}),
    variants=cfg_get(config, "model.feature_ablation.active_variants", []),
    modes=cfg_get(config, "model.training.modes", {}),
    active_modes=cfg_get(config, "model.training.active_modes", []),
    n_splits=int(cfg_get(config, "validation.n_folds", 5)),
    fast=bool(cfg_get(config, "audit.fast", False)),
    early_stopping_rounds=int(cfg_get(config, "model.training.early_stopping_rounds", 250)),
    max_rows=cfg_get(config, "model.training.max_rows"),
    max_train_rows=cfg_get(config, "model.training.max_train_rows"),
    save_models=bool(cfg_get(config, "model.training.save_models", True)),
    save_predictions=bool(cfg_get(config, "model.training.save_predictions", True)),
    top_n_importance=int(cfg_get(config, "model.training.top_n_importance", 60)),
)
paths.metrics_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True) + "\n")
print(json.dumps({
    "status": summary["status"],
    "dense_feature_source": summary["dense_feature_source"],
    "best_lgb_mean_by_rmse_tvt": summary["best_lgb_mean_by_rmse_tvt"],
}, indent=2, ensure_ascii=False)[:4000])
print("Metrics written:", paths.metrics_path)


## 4. Metrics and generated artifacts

In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_metrics.csv")
by_well = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_by_well.csv")
bucket_metrics = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_bucket_metrics.csv")
projection_summary = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_projection_feature_summary.csv")
dense_summary = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_dense_feature_summary.csv")
importance_mean = pd.read_csv(paths.artifacts_dir / f"{OUTPUT_PREFIX}_feature_importance_mean.csv")
manifest_path = paths.artifacts_dir / f"{OUTPUT_PREFIX}_lgb_models" / "manifest.json"

pooled = metrics[metrics["fold"].astype(str).eq("pooled")].sort_values("rmse_tvt")
display(pooled)
display(dense_summary)
display(projection_summary)
display(bucket_metrics.head(50))
display(by_well.head(30))
display(importance_mean.head(60))
print("Model manifest:", manifest_path, "exists=", manifest_path.exists())
print("Feature importance plot:", paths.artifacts_dir / f"{OUTPUT_PREFIX}_feature_importance_mean_top.png")
